In [2]:
# Install transformers if not already installed
# !pip install transformers

import pandas as pd
import numpy as np
# from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os

# Load IndoBERTweet tokenizer and model
# tokenizer = AutoTokenizer.from_pretrained("indolem/indobertweet-base-uncased")
# model = AutoModel.from_pretrained("indolem/indobertweet-base-uncased")

# Load your data
df = pd.read_json('fetched_data_final_dedup.json')
X = df['text'].tolist()
y = df['label'].astype(int).tolist()

In [ ]:

# Function to get sentence embeddings
def get_bertweet_embeddings(texts, tokenizer, model, max_length=128, batch_size=16, save_path=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)   # move model to GPU
    model.eval()
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
            # move inputs to GPU
            encoded = {key: val.to(device) for key, val in encoded.items()}
            outputs = model(**encoded)
            # Use [CLS] token embedding as sentence representation
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
    result = np.vstack(embeddings)

    if save_path is not None:
        # ensure directory exists
        save_dir = f"/kaggle/working/{save_path}"
        os.makedirs(save_dir, exist_ok=True)
        np.save(f"{save_dir}/embeddings_{max_length}.npy", result)
        print(f"Embeddings saved to {save_dir}/embeddings_{max_length}.npy")

    return result

In [ ]:
X_embeddings_128 = get_bertweet_embeddings(X, tokenizer, model, max_length=128)
X_embeddings_256 = get_bertweet_embeddings(X, tokenizer, model, max_length=256)
X_embeddings_512 = get_bertweet_embeddings(X, tokenizer, model, max_length=512)


In [ ]:
from sklearn.model_selection import StratifiedKFold

# Get IndoBERTweet embeddings for all texts
X_embeddings = np.load("./embed/embeddings_128.npy")
# X_embeddings = np.load("./embed/embeddings_256.npy") --- IGNORE ---
# X_embeddings = np.load("./embed/embeddings_512.npy") --- IGNORE --
n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
all_reports = []
all_y_true = []
all_y_pred = []
cv_scores = []
fold_reports = []
fold = 0

for fold, (train_idx, test_idx) in enumerate(skf.split(X_embeddings, y)):
    X_train, X_test = X_embeddings[train_idx], X_embeddings[test_idx]
    y_train, y_test = np.array(y)[train_idx], np.array(y)[test_idx]

    rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    accu = accuracy_score(y_test, y_pred)
    cv_scores.append(accu)
    print("Fold {}: {}".format(fold + 1, accu))
    # Collect metrics for this fold
    report = classification_report(y_test, y_pred, digits=4, output_dict=True, zero_division=0)
    fold_reports.append(report)
    display(pd.DataFrame(report).transpose())
    fold += 1

labels = sorted(list(set(y)))
metrics = ['precision', 'recall', 'f1-score']
mean_results = {}


# Per-class averages
for label in labels:
    mean_results[f"class {label}"] = {
        m: np.mean([fold[str(label)][m] for fold in fold_reports]) for m in metrics
    }

# Global averages
mean_results['accuracy'] = {"score": np.mean([fold['accuracy'] for fold in fold_reports])}
mean_results['macro avg'] = {
    m: np.mean([fold['macro avg'][m] for fold in fold_reports]) for m in metrics
}
mean_results['weighted avg'] = {
    m: np.mean([fold['weighted avg'][m] for fold in fold_reports]) for m in metrics
}

# Convert to DataFrame
results_df = pd.DataFrame(mean_results).T

print("Final averaged metrics across folds:")
display(results_df)

# Show per-fold accuracies as well
fold_acc_df = pd.DataFrame({
    "Fold": range(1, n_splits + 1),
    "Accuracy": cv_scores
})
print("Per-fold accuracy:")
display(fold_acc_df)

# Show overall confusion matrix
print("Confusion Matrix across all folds:")
display(pd.DataFrame(confusion_matrix(all_y_true, all_y_pred)))



Fold 1: 0.8927637314734089


,precision,recall,f1-score,support
0,0.862355,0.928571,0.894239,560.000000
1,0.926471,0.858603,0.891247,587.000000
accuracy,0.892764,0.892764,0.892764,0.892764
macro avg,0.894413,0.893587,0.892743,1147.000000
weighted avg,0.895167,0.892764,0.892708,1147.000000


Fold 2: 0.9058413251961639


,precision,recall,f1-score,support
0,0.884354,0.928571,0.905923,560.000000
1,0.928444,0.884157,0.905759,587.000000
accuracy,0.905841,0.905841,0.905841,0.905841
macro avg,0.906399,0.906364,0.905841,1147.000000
weighted avg,0.906918,0.905841,0.905839,1147.000000


Fold 3: 0.8927637314734089


...

,precision,recall,f1-score,score
class 0,0.876841,0.921701,0.898657,NaN
class 1,0.921717,0.876614,0.898540,NaN
accuracy,NaN,NaN,NaN,0.898605
macro avg,0.899279,0.899158,0.898598,NaN
weighted avg,0.899829,0.898605,0.898597,NaN


Per-fold accuracy:


,Fold,Accuracy
0,1,0.892764
1,2,0.905841
2,3,0.892764
3,4,0.897123
4,5,0.901482
5,6,0.883173
6,7,0.906713
7,8,0.885789
8,9,0.915432
9,10,0.904969


Confusion Matrix across all folds:


,0,1
0,5156,438
1,725,5151


In [1]:
import pandas as pd
import numpy as np
import json
import joblib
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, make_scorer, f1_score, precision_score, recall_score
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Load data from JSON file (array of objects)
with open('fetched_data_final_dedup.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
df['label'] = df['label'].astype(int)

X = df['text']
y = df['label']

# load embedding from .npy file
X_embeddings = np.load("./embed/embeddings_128.npy")
# X_embeddings = np.load("./embed/embeddings_256.npy") --- IGNORE ---
# X_embeddings = np.load("./embed/embeddings_512.npy") --- IGNORE ---

# Choose number of splits (5 or 10)
n_splits = 10  # Change to 5 or 10 as needed
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Define parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 50],
    'min_samples_split': [2, 5, 10],
    'class_weight': [None, 'balanced']
}

# Define scoring metrics
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'f1': make_scorer(f1_score, average='weighted'),
    'precision': make_scorer(precision_score, average='weighted'),
    'recall': make_scorer(recall_score, average='weighted')
}

# Grid Search with StratifiedKFold
search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=skf,
    scoring=scoring,
    refit='f1',  # Optimize for weighted F1-score
    n_jobs=-1,
    verbose=2
)

search.fit(X_embeddings, y)

print("Best parameters:", search.best_params_)
print("Best cross-validated score:", search.best_score_)

import pandas as pd

results = pd.DataFrame(search.cv_results_)
print(results[['param_n_estimators', 'mean_test_accuracy', 'mean_test_f1', 'mean_test_precision', 'mean_test_recall']])



Fitting 10 folds for each of 72 candidates, totalling 720 fits
Best parameters: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 300}
Best cross-validated score: 0.9041848931865308
    param_n_estimators  mean_test_accuracy  mean_test_f1  mean_test_precision  \
0                  100            0.899738      0.899732             0.900876   
1                  200            0.901831      0.901831             0.902786   
2                  300            0.901656      0.901656             0.902611   
3                  100            0.900959      0.900963             0.901733   
4                  200            0.900349      0.900351             0.901242   
..                 ...                 ...           ...                  ...   
67                 200            0.902528      0.902526             0.903566   
68                 300            0.903923      0.903923             0.904874   
69                 100            0.900087      0.9

In [2]:
# SVM Classifier with IndoBERTweet Embeddings
from sklearn.svm import SVC

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
# Get IndoBERTweet embeddings for all texts (reuse if already computed)
X_embeddings = np.load("./embed/embeddings_128.npy")
n_splits = 10
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
all_reports = []
all_y_true = []
all_y_pred = []
cv_scores = []
fold_reports = []
fold = 0

for fold, (train_idx, test_idx) in enumerate(skf.split(X_embeddings, y)):
    X_train, X_test = X_embeddings[train_idx], X_embeddings[test_idx]
    y_train, y_test = np.array(y)[train_idx], np.array(y)[test_idx]

    # Initialize and train SVM (RBF kernel as default, can tune parameters)
    svm = SVC(kernel='rbf', C=1.0, class_weight='balanced', random_state=42)
    svm.fit(X_train, y_train)
    # Predict and evaluate
    y_pred = svm.predict(X_test)
    
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    accu = accuracy_score(y_test, y_pred)
    cv_scores.append(accu)
    print("Fold {}: {}".format(fold + 1, accu))
    # Collect metrics for this fold
    report = classification_report(y_test, y_pred, digits=4, output_dict=True, zero_division=0)
    fold_reports.append(report)
    display(pd.DataFrame(report).transpose())
    # fold += 1

labels = sorted(list(set(y)))
metrics = ['precision', 'recall', 'f1-score']
mean_results = {}


# Per-class averages
for label in labels:
    mean_results[f"class {label}"] = {
        m: np.mean([fold[str(label)][m] for fold in fold_reports]) for m in metrics
    }

# Global averages
mean_results['accuracy'] = {"score": np.mean([fold['accuracy'] for fold in fold_reports])}
mean_results['macro avg'] = {
    m: np.mean([fold['macro avg'][m] for fold in fold_reports]) for m in metrics
}
mean_results['weighted avg'] = {
    m: np.mean([fold['weighted avg'][m] for fold in fold_reports]) for m in metrics
}

# Convert to DataFrame
results_df = pd.DataFrame(mean_results).T

print("Final averaged metrics across folds:")
display(results_df)

# Show per-fold accuracies as well
fold_acc_df = pd.DataFrame({
    "Fold": range(1, n_splits + 1),
    "Accuracy": cv_scores
})
print("Per-fold accuracy:")
display(fold_acc_df)

# Show overall confusion matrix
print("Confusion Matrix across all folds:")
display(pd.DataFrame(confusion_matrix(all_y_true, all_y_pred)))


Fold 1: 0.9337401918047079


,precision,recall,f1-score,support
0,0.923077,0.942857,0.932862,560.00000
1,0.944348,0.925043,0.934596,587.00000
accuracy,0.933740,0.933740,0.933740,0.93374
macro avg,0.933712,0.933950,0.933729,1147.00000
weighted avg,0.933963,0.933740,0.933749,1147.00000


Fold 2: 0.9224062772449869


,precision,recall,f1-score,support
0,0.903945,0.941071,0.922135,560.000000
1,0.941489,0.904600,0.922676,587.000000
accuracy,0.922406,0.922406,0.922406,0.922406
macro avg,0.922717,0.922836,0.922405,1147.000000
weighted avg,0.923159,0.922406,0.922412,1147.000000


Fold 3: 0.9250217959895379


...

,precision,recall,f1-score,score
class 0,0.920132,0.946017,0.932838,NaN
class 1,0.947295,0.921714,0.934271,NaN
accuracy,NaN,NaN,NaN,0.933566
macro avg,0.933713,0.933866,0.933554,NaN
weighted avg,0.934048,0.933566,0.933572,NaN


Per-fold accuracy:


,Fold,Accuracy
0,1,0.933740
1,2,0.922406
2,3,0.925022
3,4,0.937228
4,5,0.945074
5,6,0.938099
6,7,0.932868
7,8,0.922406
8,9,0.945074
9,10,0.933740


Confusion Matrix across all folds:


,0,1
0,5292,302
1,460,5416
